In [1]:
import os
import pandas as pd

# --- Configuração (ajuste os caminhos) ---
INPUT_PATH = "../data/interim/01b_data_after_yfinance.csv"   # tabela de saída antiga
CACHE_PATH = "../data/interim/yf_cache.csv"                  # arquivo do cache

COLUNAS_YF = [
    "ticker_symbol", "website", "longBusinessSummary",
    "sector", "industry", "country", "city", "yf_status"
]

# --- 0. Ler a tabela de origem ---
df_cache = pd.read_csv(INPUT_PATH)

# --- 1. Filtrar: resumo realmente preenchido + origem yfinance ---
tem_resumo = df_cache["longBusinessSummary"].fillna("").str.strip() != ""
mascara = tem_resumo & (df_cache["yf_status"] == "success")

print(f"Com resumo: {tem_resumo.sum()} | Com resumo + success: {mascara.sum()}")

# --- 2. Montar o cache novo (1 linha por ticker) ---
cache_novo = (
    df_cache[mascara]
    .dropna(subset=["ticker_symbol"])
    .drop_duplicates(subset="ticker_symbol")
    [COLUNAS_YF]
    .copy()
)
cache_novo["yf_cached_at"] = pd.Timestamp.now().normalize()

# --- 3. Combinar com cache existente (idempotente) ---
if os.path.exists(CACHE_PATH):
    cache_antigo = pd.read_csv(CACHE_PATH)
    cache = pd.concat([cache_antigo, cache_novo], ignore_index=True)
else:
    cache = cache_novo

cache = cache.drop_duplicates(subset="ticker_symbol", keep="last")

# --- 4. Salvar ---
cache.to_csv(CACHE_PATH, index=False)
print(f"Cache salvo: {len(cache)} tickers em {CACHE_PATH}")

Com resumo: 10726 | Com resumo + success: 10715
Cache salvo: 10706 tickers em ../data/interim/yf_cache.csv


In [ ]:
from IPython.display import display

# check for empty lines in each column

df = pd.read_csv(CACHE_PATH)

for col in df.columns:
    empty_count = df[col].isnull().sum() + (df[col].astype(str).str.strip() == "").sum()
    print(f"Coluna '{col}': {empty_count} linhas vazias")

# show full line of empty sector and industry

empty_sector_industry = df[(df['sector'].isnull() | (df['sector'].astype(str).str.strip() == "")) &
                           (df['industry'].isnull() | (df['industry'].astype(str).str.strip() == ""))]

print("\nLinhas com 'sector' e 'industry' vazios:")
display(empty_sector_industry)


Coluna 'ticker_symbol': 0 linhas vazias
Coluna 'website': 0 linhas vazias
Coluna 'longBusinessSummary': 0 linhas vazias
Coluna 'sector': 23 linhas vazias
Coluna 'industry': 23 linhas vazias
Coluna 'country': 0 linhas vazias
Coluna 'city': 1 linhas vazias
Coluna 'yf_status': 0 linhas vazias
Coluna 'yf_cached_at': 0 linhas vazias

Linhas com 'sector' e 'industry' vazios:


,ticker_symbol,website,longBusinessSummary,sector,industry,country,city,yf_status,yf_cached_at
293,1BF.SG,https://www.standardlifeplc.com,Standard Life plc operates in the long-term sa...,NaN,NaN,United Kingdom,London,success,2026-10-10
1438,KKS.F,https://ir.kaspi.kz,"Joint Stock Company Kaspi.kz, together with it...",NaN,NaN,Kazakhstan,Almaty,success,2026-10-10
2633,ABAM1.JO,https://www.absa.co.za/personal,"Absa Bank Limited, together with its subsidiar...",NaN,NaN,South Africa,Johannesburg,success,2026-10-10
5050,0N0L.L,https://bewiinvest.com,"BEVEST ASA operates as an investment firm, pri...",NaN,NaN,Norway,Trondheim,success,2026-10-10
5746,4ZQ.F,https://www.tritaxbigbox.co.uk,Tritax Big Box REIT plc is the largest listed ...,NaN,NaN,United Kingdom,London,success,2026-10-10
6617,TINPLATE.NS,https://www.tatatinplate.com,"As of January 15, 2024.The Tinplate Company of...",NaN,NaN,India,Kolkata,success,2026-10-10
7362,0QQI.L,https://www.centiel.com,"Centiel AG designs, manufactures, and supplies...",NaN,NaN,Switzerland,Lugano,success,2026-10-10
7421,WUG3.F,https://www.westag.de,Westag AG manufactures and sells wood-based pr...,NaN,NaN,Germany,Rheda-Wiedenbrück,success,2026-10-10
7582,SKAGI.IC,https://skagi.is,Skagi hf. provides various insurance products ...,NaN,NaN,Iceland,Reykjavik,success,2026-10-10
8427,YOUNI.PA,https://www.younited.com,Younited Financial S.A. provides consumer cred...,NaN,NaN,Luxembourg,Luxembourg,success,2026-10-10
